# Cart sweep → pendulum damping coefficient (b)

One notebook does everything: **runs the sweep on the Arduino → saves a new raw CSV → fits the damping → saves a new CSV with the b values.**

**Order of use:** upload `cart_sweep.ino` once → close the Arduino Serial Monitor → set the config cell → run cells top to bottom.
Cart at the start end of the track, pendulum hanging still.

Every run gets its own timestamp, so nothing is overwritten:
- `sweep_<timestamp>.csv`: raw measurements (distance vs peak angle)
- `damping_<timestamp>.csv`: per-distance measured vs model angle, plus the fitted **b** values
- `damping_fit_<timestamp>.png`: the fit plot

Model: θ'' + (mℓ/I)·cos θ · x'' + ω₀² sin θ + 2ζω₀ θ' = 0, so the damping torque is **τ = −b·θ'** with **b = 2ζω₀·I**.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import serial                      # pip install pyserial  (if this fails, run: pip install pyserial, then restart the kernel)

## 1. Config (edit these)

In [ ]:
RUN_ID = time.strftime("%Y%m%d_%H%M%S")      # new timestamp every time this cell runs -> new files each run

# ---- Serial ----
PORT = "COM8"
BAUD = 115200

# ---- Cart (same values as your sweep notebook) ----
V_MAX_MM_S  = 163.43
A_MAX_MM_S2 = 2621.03
MOTOR_STEPS_PER_REV = 1600.0       # must match the .ino
STEPS_PER_MM = 43.24               # verify with a ruler: RUN,2000 should move ~46.3 mm
MAX_TRAVEL_STEPS = 10810           # must match MAX_TRAVEL_STEPS in the .ino

VMAX_RPM    = V_MAX_MM_S  * STEPS_PER_MM * 60.0 / MOTOR_STEPS_PER_REV
ACCEL_RPM_S = A_MAX_MM_S2 * STEPS_PER_MM * 60.0 / MOTOR_STEPS_PER_REV
RETURN_RPM, RETURN_ACCEL_RPM_S = VMAX_RPM, ACCEL_RPM_S
WINDOW_MS  = 2000                  # watch time after the stop (experiment AND model)
COOLDOWN_S = 15                    # pause between trials (0 = none)

# ---- Sweep ----
D_START_MM = 5
D_STOP_MM  = 180
N_POINTS   = 13
REPEATS    = 2
SHUFFLE    = False

# ---- Pendulum ----
OMEGA0 = 9.44                      # natural frequency [rad/s] (use your measured value)
M_BOB  = 0.100                     # [kg] mass at the hanging end   <-- EDIT
L_PEND = 0.250                     # [m]  pivot to the mass         <-- EDIT
I_PEND = M_BOB * L_PEND**2         # point mass at the end: I = m L^2
FIT_OMEGA0 = False                 # True = also fit omega0
G = 9.81

# ---- Skip the hardware? Set this to an existing sweep CSV to re-fit it instead of running a new sweep ----
LOAD_CSV = None                    # e.g. "sweep_20260101_120000.csv"   (None = run a new sweep)

A_MAX = A_MAX_MM_S2 / 1000.0
V_MAX = V_MAX_MM_S  / 1000.0
print(f"Run ID: {RUN_ID}   |   VMAX_RPM = {VMAX_RPM:.1f}, ACCEL_RPM_S = {ACCEL_RPM_S:.0f}")

## 2. Serial helpers

In [ ]:
def _read_until(ser, prefixes, timeout_s):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        line = ser.readline().decode(errors="ignore").strip()
        if not line:
            continue
        if line.startswith("ERR"):
            raise RuntimeError(f"Arduino reported: {line}")
        if line.startswith(prefixes):
            return line
    raise TimeoutError(f"No {prefixes} reply within {timeout_s} s")


def _send(ser, text, expect, timeout_s=5):
    ser.write((text + "\n").encode())
    return _read_until(ser, expect, timeout_s)


def run_sweep(distances_mm, repeats=1, shuffle=False, trial_timeout_s=180, csv_path=None):
    """Run every distance on the Arduino, save the CSV after every trial, return a DataFrame."""
    longest_steps = max(distances_mm) * STEPS_PER_MM
    if longest_steps > MAX_TRAVEL_STEPS:
        raise ValueError(f"{max(distances_mm)} mm = {longest_steps:.0f} steps > MAX_TRAVEL_STEPS ({MAX_TRAVEL_STEPS}).")

    plan = [(float(d), r) for r in range(repeats) for d in distances_mm]
    if shuffle:
        np.random.shuffle(plan)

    rows = []
    ser = serial.Serial(PORT, BAUD, timeout=1)
    try:
        time.sleep(2.0)                       # Arduino resets when the port opens
        ser.reset_input_buffer()
        _send(ser, "PING", "READY")

        for key, val in [("VMAX_RPM", VMAX_RPM), ("ACCEL_RPM_S", ACCEL_RPM_S), ("WINDOW_MS", WINDOW_MS),
                         ("RETURN_RPM", RETURN_RPM), ("RETURN_ACCEL_RPM_S", RETURN_ACCEL_RPM_S)]:
            _send(ser, f"SET,{key},{val}", "OK")

        for i, (d_mm, rep) in enumerate(plan, 1):
            steps = int(round(d_mm * STEPS_PER_MM))
            t0 = time.time()
            reply = _send(ser, f"RUN,{steps}", "RESULT", timeout_s=trial_timeout_s)
            _, st, pos, neg, amp, settled = reply.split(",")

            rows.append(dict(D_mm=int(st) / STEPS_PER_MM, steps=int(st), repeat=rep,
                             peak_pos_deg=float(pos), peak_neg_deg=float(neg),
                             amp_deg=float(amp), settled=bool(int(settled))))
            pd.DataFrame(rows).to_csv(csv_path, index=False)      # save after every trial
            flag = "" if rows[-1]["settled"] else "  (WARNING: pendulum did not fully settle)"
            print(f"[{i}/{len(plan)}] D = {d_mm:6.1f} mm -> amp = {amp} deg  ({time.time()-t0:.0f} s){flag}")
            if i < len(plan):
                time.sleep(COOLDOWN_S)
    finally:
        ser.close()

    print(f"Saved raw data: {csv_path}")
    return pd.DataFrame(rows).sort_values(["D_mm", "repeat"]).reset_index(drop=True)

## 3. Get the data (runs the sweep, or loads `LOAD_CSV`)

In [ ]:
SWEEP_CSV = f"sweep_{RUN_ID}.csv"

if LOAD_CSV:
    data = pd.read_csv(LOAD_CSV)
    print(f"Loaded {LOAD_CSV}")
else:
    distances_mm = np.linspace(D_START_MM, D_STOP_MM, N_POINTS)
    data = run_sweep(distances_mm, repeats=REPEATS, shuffle=SHUFFLE, csv_path=SWEEP_CSV)
data

## 4. Model (cart profile + pendulum ODE)

In [ ]:
def motion_profile(D):
    d_ramp = V_MAX**2 / A_MAX
    if D >= d_ramp:
        t_acc, t_cruise = V_MAX / A_MAX, (D - d_ramp) / V_MAX
    else:
        t_acc, t_cruise = np.sqrt(A_MAX * D) / A_MAX, 0.0
    return dict(t_acc=t_acc, t_cruise=t_cruise, T=2 * t_acc + t_cruise)


def cart_accel(t, p):
    if t < p["t_acc"]:                  return  A_MAX
    if t < p["t_acc"] + p["t_cruise"]:  return  0.0
    if t < p["T"]:                      return -A_MAX
    return 0.0


def sim_peak_deg(D_mm, zeta, omega0):
    """Peak |theta| [deg] within WINDOW_MS after the cart stops."""
    p = motion_profile(D_mm / 1000.0)
    k = omega0**2 / G                                   # = m*l/I
    def f(t, s):
        th, om = s
        xdd = cart_accel(t, p)
        return [om, -k*np.cos(th)*xdd - omega0**2*np.sin(th) - 2*zeta*omega0*om]
    t_end = p["T"] + WINDOW_MS / 1000.0
    t = np.arange(0, t_end, 0.002)
    sol = solve_ivp(f, (0, t_end), [0.0, 0.0], t_eval=t, max_step=0.005, rtol=1e-7, atol=1e-9)
    return np.degrees(np.max(np.abs(sol.y[0][sol.t >= p["T"]])))

## 5. Fit ζ and compute b

In [ ]:
good = data[data["settled"]] if "settled" in data else data      # drop trials that didn't settle
if good.empty:
    raise RuntimeError("No settled trials to fit. Check the rig / increase the settle time.")
meas = good.groupby("D_mm")["amp_deg"].mean()                     # average repeats
D_meas, A_meas = meas.index.values, meas.values


def residuals(params):
    zeta = params[0]
    w0 = params[1] if FIT_OMEGA0 else OMEGA0
    return np.array([sim_peak_deg(d, zeta, w0) for d in D_meas]) - A_meas

x0 = [0.02] + ([OMEGA0] if FIT_OMEGA0 else [])
lo = [0.0]  + ([0.8 * OMEGA0] if FIT_OMEGA0 else [])
hi = [0.5]  + ([1.2 * OMEGA0] if FIT_OMEGA0 else [])
fit = least_squares(residuals, x0, bounds=(lo, hi), diff_step=1e-3)

zeta_fit = fit.x[0]
w0_fit = fit.x[1] if FIT_OMEGA0 else OMEGA0

dof = max(len(D_meas) - len(fit.x), 1)
s2 = np.sum(fit.fun**2) / dof
try:
    zeta_err = np.sqrt((np.linalg.inv(fit.jac.T @ fit.jac) * s2)[0, 0])
except np.linalg.LinAlgError:
    zeta_err = np.nan

b_over_I = 2 * zeta_fit * w0_fit                  # [1/s]
b_torque = b_over_I * I_PEND                      # [N*m*s/rad]   torque = -b * theta_dot
tau = 1 / (zeta_fit * w0_fit) if zeta_fit > 0 else np.inf
rms = np.sqrt(np.mean(fit.fun**2))

print(f"zeta            = {zeta_fit:.4f}  (+/- {zeta_err:.4f})")
print(f"omega0          = {w0_fit:.3f} rad/s")
print(f"b / I           = {b_over_I:.4f} 1/s            (= 2*zeta*omega0)")
print(f"b  (I = mL^2 = {I_PEND:.4g} kg m^2) = {b_torque:.4g} N*m*s/rad")
print(f"amp decay time  = {tau:.1f} s   (amplitude falls to 1/e)")
print(f"RMS misfit      = {rms:.2f} deg")

## 6. Save the b values to a new CSV and plot

In [ ]:
model_deg = fit.fun + A_meas                       # model prediction at each measured distance
out = pd.DataFrame({
    "D_mm": D_meas,
    "measured_amp_deg": A_meas,
    "model_amp_deg": model_deg,
    "zeta": zeta_fit,
    "zeta_err": zeta_err,
    "omega0_rad_s": w0_fit,
    "b_over_I_1_s": b_over_I,
    "b_N_m_s_per_rad": b_torque,
    "decay_time_s": tau,
    "rms_misfit_deg": rms,
    "M_BOB_kg": M_BOB,
    "L_PEND_m": L_PEND,
    "I_kg_m2": I_PEND,
    "sweep_file": LOAD_CSV or SWEEP_CSV,
})
DAMPING_CSV = f"damping_{RUN_ID}.csv"
out.to_csv(DAMPING_CSV, index=False)
print(f"Saved b values: {DAMPING_CSV}")

Dg = np.linspace(D_meas.min(), D_meas.max(), 60)
plt.figure(figsize=(8, 5))
plt.plot(D_meas, A_meas, "D", color="C3", label="experiment")
plt.plot(Dg, [sim_peak_deg(d, zeta_fit, w0_fit) for d in Dg], "-", color="C0",
         label=f"model, zeta = {zeta_fit:.4f}")
plt.xlabel("target distance D [mm]")
plt.ylabel("peak |theta| after stop [deg]")
plt.title(f"Fitted damping vs measured sweep  (b = {b_torque:.3g} N·m·s/rad)")
plt.grid(alpha=0.3); plt.legend()
plt.savefig(f"damping_fit_{RUN_ID}.png", dpi=150, bbox_inches="tight")
plt.show()
out.head()

## Notes
- Damping only weakly changes the peak-within-window data, so check the ± on ζ. If it is large compared with ζ, the fit isn't well constrained.
- To re-fit an old run without the hardware, set `LOAD_CSV = "sweep_<timestamp>.csv"` in the config cell. A new `damping_<timestamp>.csv` is still created.